# From Go summaries to a Python notebook

SPDX-License-Identifier: Apache-2.0  
Code authors: Vijay Erramilli and Codex

**Audience:** data engineers, data scientists, and platform engineers working with high-cardinality LLM or agent telemetry.

**Prerequisites:** Python 3.11+, Go 1.25+, basic pandas, and the packages listed in this example's README.

By the end, you will be able to:

- produce keyed HLL++ and weighted frequent-items summaries in Go;
- load and validate the canonical wire bytes in Python;
- reject an incompatible merge before combining data;
- merge compatible service shards by window; and
- plot a distinct-count estimate and deterministic heavy-item bounds without moving raw identifiers into the notebook.


## Outline

1. Produce two windows of service-local summaries in Go.
2. Confirm that Python round-trips the Go wire bytes exactly.
3. Reject an incompatible profile and merge compatible shards.
4. Compare HLL++ estimates with synthetic validation counts.
5. Plot deterministic token-volume bounds for pseudonymous keys.
6. State what the results do and do not establish.


In [ ]:
# SPDX-License-Identifier: Apache-2.0
# Code authors: Vijay Erramilli and Codex
from __future__ import annotations

import json
import os
import secrets
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from llm_sketchkit import frequentitems, hllpp

get_ipython().run_line_magic("matplotlib", "inline")


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (
            (candidate / "go.mod").is_file()
            and (candidate / "pyproject.toml").is_file()
        ):
            return candidate
    raise RuntimeError("start JupyterLab from the llm-sketchkit repository")


ROOT = find_repo_root(Path.cwd().resolve())
EXAMPLE = ROOT / "examples" / "go-to-python"
GENERATED = EXAMPLE / "generated"
print("Ready to produce Go summaries.")


## 1. Produce bounded summaries in Go

The producer creates two service shards for each of two time windows. It sees synthetic user IDs, canonicalizes and keyed-hashes them, and updates:

- an HLL++ sketch for distinct users; and
- a weighted frequent-items sketch using reported tokens as weights.

The emitted files contain canonical sketch state, not the raw synthetic IDs. If no deployment secret is already set, this cell creates an ephemeral one for the demo without printing it.


In [ ]:
env = os.environ.copy()
env.setdefault("LLM_SKETCHKIT_SECRET", secrets.token_hex(32))

run = subprocess.run(
    [
        "go",
        "run",
        "./examples/go-to-python/producer",
        "-out",
        str(GENERATED.relative_to(ROOT)),
    ],
    cwd=ROOT,
    env=env,
    check=True,
    capture_output=True,
    text=True,
)
print(run.stdout.strip())


The manifest carries bounded operational metadata and file names. `synthetic-validation.json` contains exact aggregate answers only so this tutorial can check the estimates. A real deployment normally does not have that exact side channel.


In [ ]:
manifest = json.loads((GENERATED / "manifest.json").read_text())
validation = json.loads((GENERATED / manifest["synthetic_validation_file"]).read_text())

shards = pd.DataFrame(manifest["shards"])
shards[["window", "service", "observed_requests", "observed_token_mass"]]


## 2. Validate the Go wire bytes in Python

Parsing and re-serializing an unchanged canonical sketch must reproduce the same bytes. This checks the actual Go-to-Python handoff before any analysis begins.


In [ ]:
first = manifest["shards"][0]
users_bytes = (GENERATED / first["users_file"]).read_bytes()
tokens_bytes = (GENERATED / first["token_items_file"]).read_bytes()

users_part = hllpp.parse(users_bytes)
tokens_part = frequentitems.parse(tokens_bytes)

round_trip = {
    "hllpp_bytes": len(users_bytes),
    "hllpp_byte_identical": users_part.marshal_binary() == users_bytes,
    "frequent_items_bytes": len(tokens_bytes),
    "frequent_items_byte_identical": tokens_part.marshal_binary() == tokens_bytes,
}
assert all(value for key, value in round_trip.items() if key.endswith("identical"))
round_trip


## 3. Reject incompatible state, then merge valid shards

Compatibility is part of the wire contract. The next cell deliberately tries to merge the `small` HLL++ profile with a `micro` profile. Python rejects it instead of silently changing precision.


In [ ]:
incompatible = hllpp.parse(
    (GENERATED / manifest["incompatible_users_file"]).read_bytes()
)

try:
    users_part.clone().merge(incompatible)
except (hllpp.PrecisionMismatchError, hllpp.IncompatibleMergeError) as exc:
    print(f"rejected as expected: {type(exc).__name__}: {exc}")
else:
    raise AssertionError("incompatible HLL++ profiles unexpectedly merged")


In [ ]:
def merge_window(window: str) -> tuple[hllpp.Sketch, frequentitems.Sketch]:
    records = [record for record in manifest["shards"] if record["window"] == window]
    if not records:
        raise ValueError(f"unknown window: {window}")

    users = hllpp.parse((GENERATED / records[0]["users_file"]).read_bytes())
    tokens = frequentitems.parse(
        (GENERATED / records[0]["token_items_file"]).read_bytes()
    )
    for record in records[1:]:
        users.merge(hllpp.parse((GENERATED / record["users_file"]).read_bytes()))
        tokens.merge(
            frequentitems.parse(
                (GENERATED / record["token_items_file"]).read_bytes()
            )
        )
    return users, tokens


windows = [row["window"] for row in validation["windows"]]
merged = {window: merge_window(window) for window in windows}
print(f"merged {len(windows)} windows from {len(manifest['shards'])} Go shards")


## 4. Inspect distinct-user estimates

The HLL++ `small` profile has a conservative characterization envelope of 2.4375% relative error for the documented test grid. It is a profile-level engineering bound, not a per-estimate confidence interval returned by the API. The exact count below exists only because this is synthetic validation data.


In [ ]:
HLL_SMALL_CHARACTERIZATION_BOUND = 0.024375
exact_by_window = {row["window"]: row for row in validation["windows"]}

summary_rows = []
for window, (users, tokens) in merged.items():
    estimate = users.estimate()
    exact = exact_by_window[window]["exact_distinct_users"]
    relative_error = abs(estimate - exact) / exact
    summary_rows.append(
        {
            "window": window,
            "distinct_estimate": estimate,
            "synthetic_exact_distinct": exact,
            "relative_error_pct": 100 * relative_error,
            "characterization_bound_pct": 100 * HLL_SMALL_CHARACTERIZATION_BOUND,
            "within_characterized_bound": relative_error
            <= HLL_SMALL_CHARACTERIZATION_BOUND,
            "reported_token_mass": tokens.total_weight(),
        }
    )

summary = pd.DataFrame(summary_rows).set_index("window")
assert summary["within_characterized_bound"].all()
summary.round(3)


In [ ]:
ax = summary[["synthetic_exact_distinct", "distinct_estimate"]].plot.bar(
    color=["#4c78a8", "#f58518"],
    figsize=(8, 4),
    rot=0,
)
ax.set_title("Distinct users: synthetic truth and HLL++ estimate")
ax.set_ylabel("users")
ax.set_xlabel("")
ax.legend(["synthetic exact", "HLL++ estimate"], frameon=False)
plt.tight_layout()
plt.show()


## 5. Plot deterministic token-volume bounds

Weighted frequent-items returns a deterministic lower and upper bound for every tracked key. The keys remain pseudonymous: the notebook can show concentration and recurrence, but it cannot name a person without a separately authorized candidate mapping and the secret.


In [ ]:
def bounded_items(window: str, limit: int = 8) -> pd.DataFrame:
    tokens = merged[window][1]
    exact = {
        item["key_hex"]: item["tokens"]
        for item in exact_by_window[window]["exact_top_token_keys"]
    }
    rows = []
    for item in tokens.frequent_items(frequentitems.NO_FALSE_NEGATIVES)[:limit]:
        key_hex = f"{item.hash:016x}"
        exact_tokens = exact.get(key_hex)
        if exact_tokens is not None:
            assert item.lower_bound <= exact_tokens <= item.upper_bound
        rows.append(
            {
                "window": window,
                "key": key_hex[:10],
                "lower_bound": item.lower_bound,
                "estimate": item.estimate,
                "upper_bound": item.upper_bound,
                "synthetic_exact": exact_tokens,
            }
        )
    return pd.DataFrame(rows)


heavy = pd.concat([bounded_items(window) for window in windows], ignore_index=True)
heavy


In [ ]:
plot_window = "window-2"
plot_data = heavy[heavy["window"] == plot_window].sort_values("estimate").copy()
plot_data["lower_pct"] = 100 * plot_data["lower_bound"] / plot_data["upper_bound"]
plot_data["exact_pct"] = 100 * plot_data["synthetic_exact"] / plot_data["upper_bound"]
y = range(len(plot_data))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hlines(
    y,
    plot_data["lower_pct"],
    100,
    color="#4c78a8",
    linewidth=6,
    label="deterministic interval",
)
ax.scatter(
    [100] * len(plot_data),
    y,
    color="#f58518",
    label="upper estimate",
    zorder=3,
)
ax.scatter(
    plot_data["exact_pct"],
    y,
    color="#54a24b",
    marker="x",
    s=55,
    label="synthetic exact",
    zorder=4,
)
ax.set_yticks(list(y), plot_data["key"])
ax.set_xlim(plot_data["lower_pct"].min() - 0.5, 100.2)
ax.set_xlabel("percent of upper estimate")
ax.set_ylabel("pseudonymous user key")
ax.set_title(f"Synthetic truth inside deterministic token bounds: {plot_window}")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Exercise: bound concentration without naming anyone

Compute the share of reported token mass attributable to the top three returned keys. Because each key has an interval, report a lower and upper share rather than one over-precise percentage.


In [ ]:
def top_k_share_bounds(tokens: frequentitems.Sketch, k: int) -> tuple[float, float]:
    items = tokens.frequent_items(frequentitems.NO_FALSE_NEGATIVES)[:k]
    total = tokens.total_weight()
    return (
        sum(item.lower_bound for item in items) / total,
        sum(item.upper_bound for item in items) / total,
    )


concentration = pd.DataFrame(
    [
        {
            "window": window,
            "top_3_lower_share_pct": 100 * top_k_share_bounds(merged[window][1], 3)[0],
            "top_3_upper_share_pct": 100 * top_k_share_bounds(merged[window][1], 3)[1],
        }
        for window in windows
    ]
).set_index("window")
concentration.round(2)


## What crossed the boundary

The notebook received canonical sketch state and bounded synthetic validation aggregates. It did not receive raw user IDs.

Important limits:

- Keyed hashes are pseudonymous, not anonymous. They remain linkable while the same secret and domain are used.
- Anyone with the secret and a candidate value can test that candidate. Protect serialized summaries and the secret separately.
- Secret rotation intentionally breaks cross-window comparability and mergeability.
- HLL++ provides an approximate distinct count with a characterized profile-level error envelope; it does not return a per-estimate confidence interval.
- Weighted frequent-items bounds are deterministic. Its serialization is deterministic for a fixed state, but arbitrary merge orders are not promised to produce byte-identical state.
- Concentration shows where reported token volume accumulated. It does not establish waste, task value, or causation.
- This is a library-to-notebook workflow. The OpenTelemetry connector currently exports metrics and bounded structured top-item records, not these serialized sketch payloads.
